In [30]:
%pip install openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import requests
from bs4 import BeautifulSoup
import re
import pandas as pd
from datetime import datetime

In [ ]:
nr_url = 'https://naturalrefrigerants.com/news/'
## works w/ base fetch
nr_article_url = 'https://naturalrefrigerants.com/news/the-middle-east-is-ready-to-scale-natural-refrigerants-says-epta-middle-east-general-manager/'
## works w/ base fetch


trane_url = 'https://www.tranetechnologies.com/en/index/news.html'
## works w/ base fetch

##trane_news_url = 'https://investors.tranetechnologies.com/news-and-events/news-releases/default.aspx'
## doesn't work w/ base fetch

trane_article_url = 'https://investors.tranetechnologies.com/news-and-events/news-releases/news-release-details/2025/Trane-Technologies-to-Acquire-Stellar-Energy-Digital-Business/default.aspx'
## works w/ base fetch


### original base fetch method

In [ ]:
def trane_fetch_method(url):
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/123.0.0.0 Safari/537.36"
        )
    }
    response = requests.get(url, headers=headers, verify=False, timeout=30)
    print(response.status_code)
    html_doc = response.text
    soup = BeautifulSoup(html_doc, 'html.parser')
    return soup

### Extract htmls

In [ ]:
def trane_url_extraction(temp_soup):
    # Find all <a> tags with class newspromo__link
    links = temp_soup.find_all("a", class_="newspromo__link")

    # Extract href attributes
    hrefs = [a['href'] for a in links if 'href' in a.attrs]

    # Keep only links that start with https:
    https_links = [link for link in hrefs if link.startswith("https:")]

### Trane article scraping method

In [ ]:
def trane_scrape_article(url):
    print(url)
    soup = fetch_method(url)

    new_data = pd.DataFrame(columns=['title','summary','dateline','newslinetext','url','source'])

    ## extract title
    title=soup.find('h3').get_text().strip()


    ## extract summary / newslinetext 
    full_text = " ".join(p.get_text().strip() for p in temp_soup.find_all('p'))
    result_text = full_text.split("About Trane Technologies")[0].strip()

    ## extract date
    dateline = None

    try:
        date_text=soup.find("span", class_="module_date-text").get_text().strip()
        dateline = datetime.strptime(date_text, '%b %d, %Y')
    except Exception:
        ## fallback to span value
        spans = soup.find_all("span", class_="value")

        for span in spans:
            text = span.get_text(strip=True)  # extract text from each span

            try:
                dateline = datetime.strptime(text, '%b %d, %Y')
                break  # exit loop if date is found
            except ValueError:
                pass
            try:
                text_clean = text.replace(" ET", "")  # remove timezone suffix
                dateline = datetime.strptime(text_clean, "%b %d, %Y %I:%M %p")
                break
            except ValueError:
                pass
    if dateline is None:
        dateline = datetime.now()

    new_row = {'title':title, 'summary':result_text, 'dateline':dateline, 'newslinetext':result_text, 'url':url, 'source':'Trane Technologies'}
    new_data = pd.concat([new_data, pd.DataFrame([new_row])], ignore_index=True)
    return new_data



# Implementation

In [ ]:
## scrape homepage url for article links

temp_soup = fetch_method(trane_url)

# Find all <a> tags with class newspromo__link
links = temp_soup.find_all("a", class_="newspromo__link")

# Extract href attributes
hrefs = [a['href'] for a in links if 'href' in a.attrs]

# Keep only links that start with https:
https_links = [link for link in hrefs if link.startswith("https:")]




## iterate through article links and scrape each articles data

article_data = pd.DataFrame(columns=['title','summary','dateline','newslinetext','url', 'source'])

for link in https_links:
    article_data = pd.concat([article_data, scrape_article(link)], ignore_index=True)

c:\Users\203156\Desktop\Pipelines\Webscraping\venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.tranetechnologies.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


200
https://investors.tranetechnologies.com/news-and-events/news-releases/news-release-details/2026/Trane-Technologies-Named-to-FORTUNE-Magazine-Worlds-Most-Admired-Companies-List-for-Fourteenth-Consecutive-Year/default.aspx


c:\Users\203156\Desktop\Pipelines\Webscraping\venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'investors.tranetechnologies.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


200
https://investors.tranetechnologies.com/news-and-events/news-releases/news-release-details/2026/Trane-Technologies-Schedules-Fourth-Quarter-2025-Earnings-Conference-Call/default.aspx


C:\Users\203156\AppData\Local\Temp\ipykernel_29808\940235265.py:43: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  new_data = pd.concat([new_data, pd.DataFrame([new_row])], ignore_index=True)
C:\Users\203156\AppData\Local\Temp\ipykernel_29808\4188280633.py:22: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  article_data = pd.concat([article_data, scrape_article(link)], ignore_index=True)
c:\Users\203156\Desktop\Pipelines\Webscraping\venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureReq

...

C:\Users\203156\AppData\Local\Temp\ipykernel_29808\940235265.py:43: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  new_data = pd.concat([new_data, pd.DataFrame([new_row])], ignore_index=True)
c:\Users\203156\Desktop\Pipelines\Webscraping\venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'investors.tranetechnologies.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


200
https://investors.tranetechnologies.com/news-and-events/news-releases/news-release-details/2025/Trane-Technologies-and-AWS-Collaborate-to-Accelerate-Energy-Efficiency-and-Building-Decarbonization-across-Amazon-Grocery-Fulfillment-Centers-in-North-America/default.aspx


C:\Users\203156\AppData\Local\Temp\ipykernel_29808\940235265.py:43: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  new_data = pd.concat([new_data, pd.DataFrame([new_row])], ignore_index=True)
c:\Users\203156\Desktop\Pipelines\Webscraping\venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'investors.tranetechnologies.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


200


C:\Users\203156\AppData\Local\Temp\ipykernel_29808\940235265.py:43: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  new_data = pd.concat([new_data, pd.DataFrame([new_row])], ignore_index=True)


In [51]:
## write to excel
article_data.to_excel('trane_news.xlsx', index=False)